# CRIMMbrane Tutorial Overview

This tutorial builds a PDB: 9QSY protein-membrane system using CRIMM-native membrane tools. The workflow starts from the raw 9QSY protomer files, orients the tetramer with `MembraneOrienter`, builds a POPC/cholesterol bilayer with `MembraneBuilder`, validates clashes, solvates the system, and adds ions.

Current lipid building uses a conformer library, so supported membrane lipids are limited to lipid types present in `crimm/Data/lipid_lib.tar.gz`. In the current library, supported entries are `POPC`, `DMPC`, `DOPC`, `DPPC`, `DSPC`, and `CHL1`. Other lipids, such as `DLPE`, require adding matching conformers or implementing native lipid coordinate generation.

This section points Python to the local CRIMM clone, clears any previously imported CRIMM modules, and imports the tools used throughout the notebook.

In [1]:
import sys

repo_root = "/home/boloyede/crimm"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

for name in list(sys.modules):
    if name == "crimm" or name.startswith("crimm."):
        del sys.modules[name]

import crimm
print(crimm.__file__)

/home/boloyede/crimm/crimm/__init__.py


This section loads the four 9QSY protomer PDB files, assigns chain IDs A-D, removes non-protein residues, and combines the chains into one tetramer model.

In [2]:
from pathlib import Path

import numpy as np
from Bio.PDB import PDBParser, PDBIO

from crimm.Modeller.MembraneBuilder import MembraneBuilder, MembraneSpec

In [3]:
charmm_gui_dir = Path("/home/boloyede/crimm/crimm/charmm_gui_9qsy")

protein_resnames = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "CYX",
    "GLN", "GLU", "GLY", "HIS", "HSD", "HSE", "HSP",
    "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR",
    "TRP", "TYR", "VAL",
}

# Orientation

This section uses `MembraneOrienter` to center and orient the 9QSY tetramer relative to an implicit membrane slab. The atom-transfer scoring mode estimates whether atoms prefer the membrane core, interface, or water-facing region.

In [4]:
from Bio.PDB import Model
from crimm.Modeller.MembraneOrienter import MembraneOrienter

def load_raw_9qsy_tetramer():
    raw_files = {
        "A": charmm_gui_dir / "9qsy_proa.pdb",
        "B": charmm_gui_dir / "9qsy_prob.pdb",
        "C": charmm_gui_dir / "9qsy_proc.pdb",
        "D": charmm_gui_dir / "9qsy_prod.pdb",
    }

    parser = PDBParser(PERMISSIVE=True, QUIET=True)
    combined_model = Model.Model(0)

    for chain_id, pdb_path in raw_files.items():
        structure = parser.get_structure(f"9qsy_{chain_id}", pdb_path)
        raw_model = structure[0]

        raw_chain = list(raw_model)[0]
        raw_chain.id = chain_id
        raw_chain.chain_type = "Polypeptide(L)"

        for residue in list(raw_chain):
            if residue.get_resname().strip() not in protein_resnames:
                raw_chain.detach_child(residue.id)

        combined_model.add(raw_chain.copy())

    return combined_model

## Membrane Specification

This section defines the membrane build settings: lipid composition, bilayer size, lipid library source, protein exclusion radius, clash checking, solvation box, and ion concentration.

In [5]:
spec = MembraneSpec(
    box_xy=(140.0, 140.0),
    box_z=156.0,
    solvation_box=(190.0, 190.0, 156.0),

    upper_leaflet=None,
    lower_leaflet=None,
    lipid_ratios={"POPC": 0.8, "CHL1": 0.2},
    area_per_lipid=126.0,

    lipid_z=19.0,
    lipid_z_scale=0.6,
    random_seed=12345,

    use_lipid_library=True,
    lipid_library_path=None,

    use_charmm_gui_head_positions=False,
    charmm_gui_head_crd=None,
    align_head_positions_to_protein_xy=False,

    protein_exclusion_radius=4.0,

    clash_check=True,
    protein_clash_cutoff=1.7,
    membrane_clash_cutoff=1.2,
    max_repack_attempts=50,

    water_solvcut=2.8,
    remove_membrane_core_waters=True,
    membrane_core_z=15.0,

    salt_concentration=0.15,
    cation="POT",
    anion="CLA",
)

## Build Bilayer

This section builds the POPC/cholesterol bilayer around the oriented tetramer. Lipid counts are estimated from the requested lipid ratios and area per lipid, and lipid conformers are placed while avoiding close contacts with the protein.

Then it collects lipid and cholesterol head-group atoms and confirms that the upper and lower leaflet head groups are placed at the expected membrane planes.

In [6]:
protein_model = load_raw_9qsy_tetramer()

orienter = MembraneOrienter(
    protein_model,
    membrane_core_z=15.0,
    interface_z=20.0,
    interface_width=3.0,
    phi_step=10.0,
    teta_step=10.0,
    shift_step=2.0,
    shift_limit=30.0,
    score_mode="atom_transfer",
    use_exposure_weights=True,
    refine=True,
    refine_factor=5,
)

orientation = orienter.orient(in_place=True)
print(orientation)

protein_coords = np.array([atom.coord for atom in protein_model.get_atoms()])
print("chains:", [(chain.id, len(chain)) for chain in protein_model])
print("protein atom count:", len(protein_coords))
print("protein center:", protein_coords.mean(axis=0))
print("protein min:", protein_coords.min(axis=0))
print("protein max:", protein_coords.max(axis=0))

MembraneOrientation(phi=0.0, teta=0.0, shift_z=0.0, score=-0.044020253972530614)
chains: [('A', 251), ('B', 251), ('C', 251), ('D', 251)]
protein atom count: 7572
protein center: [-8.33752272e-13 -1.98377676e-13 -6.84343115e-14]
protein min: [-37.89133504 -37.89210068 -28.88393454]
protein max: [37.89766643 37.89089645 26.47806497]


In [ ]:
builder = MembraneBuilder(protein_model, spec)
builder.prepare_topology()
builder.add_lipid_bilayer()

print("phospholipids:", builder.count_lipids())
print("cholesterol:", builder.count_sterols())
print("total membrane residues:", builder.count_lipids() + builder.count_sterols())

In [ ]:
heads = []
for chain in protein_model:
    if getattr(chain, "chain_type", None) not in {"Lipid", "Sterol"}:
        continue
    for residue in chain:
        for atom in residue:
            atom_name = atom.get_name().strip().upper()
            if atom_name in {"P", "O3"}:
                heads.append((residue.resname, atom_name, atom.coord.copy()))

upper = np.array([coord for _, _, coord in heads if coord[2] > 0])
lower = np.array([coord for _, _, coord in heads if coord[2] < 0])
head_coords = np.array([coord for _, _, coord in heads])

print("total heads:", len(heads))
print("upper heads:", len(upper), "z range:", upper[:, 2].min(), upper[:, 2].max())
print("lower heads:", len(lower), "z range:", lower[:, 2].min(), lower[:, 2].max())
print("head center:", head_coords.mean(axis=0))

## Clash Validation

This section checks for close protein-membrane and lipid-lipid contacts. These checks confirm that severe overlaps were removed before solvation.

In [ ]:
protein_heavy_coords = np.array([
    atom.coord
    for chain in protein_model
    if getattr(chain, "chain_type", None) not in {"Lipid", "Sterol", "Solvent", "Ion"}
    for atom in chain.get_atoms()
    if atom.element != "H" and not atom.get_name().strip().upper().startswith("H")
])

membrane_heavy_coords = np.array([
    atom.coord
    for chain in protein_model
    if getattr(chain, "chain_type", None) in {"Lipid", "Sterol"}
    for atom in chain.get_atoms()
    if atom.element != "H" and not atom.get_name().strip().upper().startswith("H")
])

dist = np.linalg.norm(protein_heavy_coords[:, None, :] - membrane_heavy_coords[None, :, :], axis=2)

print("minimum protein-membrane distance:", dist.min())
print("contacts < 1.7 A:", np.sum(dist < 1.7))
print("contacts < 2.0 A:", np.sum(dist < 2.0))

In [ ]:
membrane_residues = []
for chain in protein_model:
    if getattr(chain, "chain_type", None) not in {"Lipid", "Sterol"}:
        continue
    for residue in chain:
        coords = []
        for atom in residue.get_atoms():
            atom_name = atom.get_name().strip().upper()
            if atom.element != "H" and not atom_name.startswith("H"):
                coords.append(atom.coord.copy())
        if coords:
            membrane_residues.append((chain.id, residue.id, residue.resname, np.array(coords)))

min_dist = float("inf")
contacts_lt_12 = 0
contacts_lt_15 = 0
worst_pairs = []

for i in range(len(membrane_residues)):
    chain_i, resid_i, resname_i, coords_i = membrane_residues[i]
    center_i = coords_i.mean(axis=0)
    for j in range(i + 1, len(membrane_residues)):
        chain_j, resid_j, resname_j, coords_j = membrane_residues[j]
        center_j = coords_j.mean(axis=0)
        if np.linalg.norm(center_i - center_j) > 15.0:
            continue
        dists = np.linalg.norm(coords_i[:, None, :] - coords_j[None, :, :], axis=2)
        local_min = float(dists.min())
        min_dist = min(min_dist, local_min)
        contacts_lt_12 += int(np.sum(dists < 1.2))
        contacts_lt_15 += int(np.sum(dists < 1.5))
        if local_min < 1.5:
            worst_pairs.append((local_min, resname_i, resid_i, resname_j, resid_j))

worst_pairs = sorted(worst_pairs, key=lambda x: x[0])[:10]
print("membrane residues checked:", len(membrane_residues))
print("minimum lipid-lipid distance:", min_dist)
print("lipid-lipid contacts < 1.2 A:", contacts_lt_12)
print("lipid-lipid contacts < 1.5 A:", contacts_lt_15)
print("worst pairs:")
for item in worst_pairs:
    print(item)

## Save And Visualize Protein + Membrane

This section saves the protein-membrane model as a PDB file and visualizes the tetramer, lipid tails, cholesterol, and leaflet head groups with `py3Dmol`.

In [ ]:
heads = []

for chain in protein_model:
    if getattr(chain, "chain_type", None) not in {"Lipid", "Sterol"}:
        continue

    for residue in chain:
        for atom in residue:
            atom_name = atom.get_name().strip().upper()
            if atom_name in {"P", "O3"}:
                heads.append((residue.resname, atom_name, atom.coord.copy()))

print("heads:", len(heads))

In [ ]:
for chain in protein_model:
    if getattr(chain, "chain_type", None) == "Lipid":
        chain.id = "M"
    elif getattr(chain, "chain_type", None) == "Sterol":
        chain.id = "Z"

out_pdb = "/home/boloyede/crimm/tetramer_native_oriented_membrane.pdb"
io = PDBIO()
io.set_structure(protein_model)
io.save(out_pdb)
print(out_pdb)

In [ ]:
import py3Dmol

view = py3Dmol.view(width=1000, height=650)
view.addModel(open(out_pdb).read(), "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "red"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "blue"}})
view.setStyle({"chain": "C"}, {"cartoon": {"color": "green"}})
view.setStyle({"chain": "D"}, {"cartoon": {"color": "purple"}})
view.setStyle({"chain": "M"}, {"line": {"color": "lime", "opacity": 0.25}})
view.setStyle({"chain": "Z"}, {"stick": {"color": "orange", "radius": 0.10, "opacity": 0.45}})
for _, _, coord in heads:
    color = "dodgerblue" if coord[2] > 0 else "crimson"
    view.addSphere({"center": {"x": float(coord[0]), "y": float(coord[1]), "z": float(coord[2])}, "radius": 0.9, "color": color, "alpha": 0.95})
view.zoomTo()
view.show()

## Solvation

This section adds water around the protein-membrane system using a solvation box that is larger than the lipid placement box. Waters inside the membrane core are removed after solvation.

Then it checks the water distribution along Z and verifies that water molecules were removed from the hydrophobic membrane core.

Lastly, it visualizes a subset of water molecules together with the protein, membrane, cholesterol, and lipid head groups.

In [ ]:
water_chains = builder.solvate()
print("water chains:", water_chains)
print("waters:", builder.count_waters())

In [ ]:
water_oxygen_coords = []

for chain in protein_model:
    if getattr(chain, "chain_type", None) != "Solvent":
        continue

    for residue in chain:
        for atom in residue:
            if atom.get_name().strip().upper() in {"OH2", "O"}:
                water_oxygen_coords.append(atom.coord.copy())

water_oxygen_coords = np.array(water_oxygen_coords)

print("water count:", len(water_oxygen_coords))
print("water z min/max:", water_oxygen_coords[:, 2].min(), water_oxygen_coords[:, 2].max())
print("waters inside membrane core:", np.sum(np.abs(water_oxygen_coords[:, 2]) < spec.membrane_core_z))
print("waters above membrane:", np.sum(water_oxygen_coords[:, 2] > spec.membrane_core_z))
print("waters below membrane:", np.sum(water_oxygen_coords[:, 2] < -spec.membrane_core_z))

In [ ]:
import py3Dmol
import numpy as np

view = py3Dmol.view(width=1000, height=650)
view.addModel(open(out_pdb).read(), "pdb")

# Protein
view.setStyle({"chain": "A"}, {"cartoon": {"color": "red"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "blue"}})
view.setStyle({"chain": "C"}, {"cartoon": {"color": "green"}})
view.setStyle({"chain": "D"}, {"cartoon": {"color": "purple"}})

# Membrane
view.setStyle({"chain": "M"}, {
    "line": {"color": "lime", "opacity": 0.25}
})
view.setStyle({"chain": "Z"}, {
    "stick": {"color": "orange", "radius": 0.10, "opacity": 0.45}
})

# Lipid/cholesterol head planes
for _, _, coord in heads:
    color = "dodgerblue" if coord[2] > 0 else "crimson"
    view.addSphere({
        "center": {
            "x": float(coord[0]),
            "y": float(coord[1]),
            "z": float(coord[2]),
        },
        "radius": 0.9,
        "color": color,
        "alpha": 0.95,
    })

# Water oxygens
water_oxygen_coords = []

for chain in protein_model:
    if getattr(chain, "chain_type", None) != "Solvent":
        continue

    for residue in chain:
        for atom in residue:
            if atom.get_name().strip().upper() in {"OH2", "O"}:
                water_oxygen_coords.append(atom.coord.copy())

water_oxygen_coords = np.array(water_oxygen_coords)

# Draw a large but manageable subset of waters.
max_waters_to_draw = 8000
step = max(1, len(water_oxygen_coords) // max_waters_to_draw)

for coord in water_oxygen_coords[::step]:
    view.addSphere({
        "center": {
            "x": float(coord[0]),
            "y": float(coord[1]),
            "z": float(coord[2]),
        },
        "radius": 0.28,
        "color": "lightskyblue",
        "alpha": 0.55,
    })

view.zoomTo()
view.show()

## Ion Placement

This section replaces selected waters with ions to neutralize the system and reach the target salt concentration. Protein charge is estimated from standard residue identities when full topology-derived charges are unavailable.

Then it counts the added ion residues and confirms the final number of ions in the solvated membrane system.

In [ ]:
builder.add_ions()

print("waters:", builder.count_waters())
print("ions:", builder.count_ions())

In [ ]:
ion_counts = {}

for chain in protein_model:
    if getattr(chain, "chain_type", None) != "Ion":
        continue

    for residue in chain:
        resname = residue.get_resname().strip()
        ion_counts[resname] = ion_counts.get(resname, 0) + 1

print(ion_counts)

In [ ]:
ion_coords = []

for chain in protein_model:
    if getattr(chain, "chain_type", None) != "Ion":
        continue

    for residue in chain:
        for atom in residue:
            ion_coords.append((residue.get_resname().strip(), atom.coord.copy()))

print("ions:", len(ion_coords))